<a href="https://colab.research.google.com/github/abhigna-web/Sudoku_solver/blob/main/HPC_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile sudoku.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

#define N 9

int isSafe(int grid[N][N], int row, int col, int num) {
    for (int x = 0; x < N; x++) {
        if (grid[row][x] == num || grid[x][col] == num) return 0;
    }
    int startRow = row - row % 3, startCol = col - col % 3;
    for (int i = 0; i < 3; i++)
        for (int j = 0; j < 3; j++)
            if (grid[i + startRow][j + startCol] == num) return 0;
    return 1;
}

int solveSudoku(int grid[N][N], int row, int col) {
    if (row == N - 1 && col == N) return 1;
    if (col == N) { row++; col = 0; }
    if (grid[row][col] != 0) return solveSudoku(grid, row, col + 1);

    int solved = 0;
    #pragma omp parallel for shared(solved)
    for (int num = 1; num <= N; num++) {
        if (!solved && isSafe(grid, row, col, num)) {
            int localGrid[N][N];
            for (int i = 0; i < N; i++)
                for (int j = 0; j < N; j++)
                    localGrid[i][j] = grid[i][j];
            localGrid[row][col] = num;
            if (solveSudoku(localGrid, row, col + 1)) {
                #pragma omp critical
                {
                    if (!solved) {
                        solved = 1;
                        for (int i = 0; i < N; i++)
                            for (int j = 0; j < N; j++)
                                grid[i][j] = localGrid[i][j];
                    }
                }
            }
        }
    }
    return solved;
}

void printGrid(int grid[N][N]) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++)
            printf("%d ", grid[i][j]);
        printf("\n");
    }
}

int main() {
    int grid[N][N] = {
        {5,3,0,0,7,0,0,0,0},
        {6,0,0,1,9,5,0,0,0},
        {0,9,8,0,0,0,0,6,0},
        {8,0,0,0,6,0,0,0,3},
        {4,0,0,8,0,3,0,0,1},
        {7,0,0,0,2,0,0,0,6},
        {0,6,0,0,0,0,2,8,0},
        {0,0,0,4,1,9,0,0,5},
        {0,0,0,0,8,0,0,7,9}
    };

    FILE *fp = fopen("sudoku_timings.csv", "w");
    fprintf(fp, "Threads,Time_seconds\n");

    // ================== THREAD NUMBERS ==================
    int thread_list[] = {1, 3, 5, 7, 9, 11, 13, 15, 16, 20, 24, 32};
    int num_tests = sizeof(thread_list) / sizeof(thread_list[0]);

    for (int t = 0; t < num_tests; t++) {
        int threads = thread_list[t];
        omp_set_num_threads(threads);          // ← UPDATED LINE (best practice)

        int tempGrid[N][N];
        for (int i = 0; i < N; i++)
            for (int j = 0; j < N; j++)
                tempGrid[i][j] = grid[i][j];

        double start = omp_get_wtime();
        if (solveSudoku(tempGrid, 0, 0)) {
            double end = omp_get_wtime();
            double time_taken = end - start;

            printf("\n=== %d THREADS ===\n", threads);
            printGrid(tempGrid);
            printf("Time taken: %.6f seconds\n", time_taken);
            fprintf(fp, "%d,%.6f\n", threads, time_taken);
        }
    }

    fclose(fp);
    printf("\n✅ Benchmark finished! \n");
    return 0;
}

Writing sudoku.c


In [ ]:
!gcc -fopenmp sudoku.c -o sudoku

In [ ]:
!./sudoku


=== 1 THREADS ===
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 4 8 5 6 
9 6 1 5 3 7 2 8 4 
2 8 7 4 1 9 6 3 5 
3 4 5 2 8 6 1 7 9 
Time taken: 0.004189 seconds

=== 3 THREADS ===
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 4 8 5 6 
9 6 1 5 3 7 2 8 4 
2 8 7 4 1 9 6 3 5 
3 4 5 2 8 6 1 7 9 
Time taken: 0.003699 seconds

=== 5 THREADS ===
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 4 8 5 6 
9 6 1 5 3 7 2 8 4 
2 8 7 4 1 9 6 3 5 
3 4 5 2 8 6 1 7 9 
Time taken: 0.003663 seconds

=== 7 THREADS ===
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 4 8 5 6 
9 6 1 5 3 7 2 8 4 
2 8 7 4 1 9 6 3 5 
3 4 5 2 8 6 1 7 9 
Time taken: 0.003671 seconds

=== 9 THREADS ===
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('sudoku_timings.csv')
threads = df['Threads']
times   = df['Time_seconds']
speedup = times[0] / times
efficiency = speedup / threads * 100

# ====================== NICER SEPARATE GRAPHS ======================

plt.rcParams.update({'font.size': 12})

# Graph 1: Execution Time (Line)
plt.figure(figsize=(9, 6))
plt.plot(threads, times, 'o-', color='blue', linewidth=2.5, markersize=8)
plt.title('Execution Time vs Number of Threads')
plt.xlabel('Number of Threads')
plt.ylabel('Time (seconds)')
plt.grid(True)
plt.xticks(threads)
plt.savefig('1_execution_time.png', dpi=300, bbox_inches='tight')
plt.show()

# Graph 2: Speedup
plt.figure(figsize=(9, 6))
plt.plot(threads, speedup, 'o-', color='green', linewidth=2.5, markersize=8)
plt.plot(threads, threads, '--', color='red', linewidth=2, label='Ideal Speedup')
plt.title('Speedup vs Number of Threads')
plt.xlabel('Number of Threads')
plt.ylabel('Speedup Factor')
plt.legend()
plt.grid(True)
plt.xticks(threads)
plt.savefig('2_speedup.png', dpi=300, bbox_inches='tight')
plt.show()

# Graph 3: Parallel Efficiency
plt.figure(figsize=(9, 6))
plt.plot(threads, efficiency, 'o-', color='purple', linewidth=2.5, markersize=8)
plt.title('Parallel Efficiency (%)')
plt.xlabel('Number of Threads')
plt.ylabel('Efficiency (%)')
plt.grid(True)
plt.xticks(threads)
plt.savefig('3_efficiency.png', dpi=300, bbox_inches='tight')
plt.show()

# Graph 4: FIXED & NICE Bar Chart (This was the bad one)
plt.figure(figsize=(10, 6))
bars = plt.bar(threads, times, color='skyblue', width=0.6, edgecolor='black', linewidth=1)
plt.title('Execution Time vs Number of Threads (Bar Chart)')
plt.xlabel('Number of Threads')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Nice value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.00005,
             f'{height:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(threads, rotation=0)
plt.tight_layout()
plt.savefig('4_execution_time_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅Graphs")
print("   → 1_execution_time.png")
print("   → 2_speedup.png")
print("   → 3_efficiency.png")
print("   → 4_execution_time_bar.png")

FileNotFoundError: [Errno 2] No such file or directory: 'sudoku_timings.csv'